# AmsiNet - Step 14: validation macro-F1 of the two word heads (CPU, about 1 minute)
Justifies the choice of the MoE head without using the test set. Runs on a CPU runtime. Paste back the printed table and the two summary lines.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import json, numpy as np, pandas as pd
from sklearn.metrics import f1_score
ROOT = '/content/drive/MyDrive/amsinet'
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8')); SIDX = {s: i for i, s in enumerate(D['scripts'])}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); d = META[META.split == 'val'].reset_index(drop=True)
y = np.array([SIDX[s] for s in d.script]); rows = []
for tag in ('word', 'word_moe'):
    for s in range(5):
        P = np.load(f'{ROOT}/agents/{tag}_seed{s}_val_probs.npy'); pa = pd.read_csv(f'{ROOT}/agents/{tag}_seed{s}_val_paths.csv').path.tolist()
        assert pa == d.path.tolist(), 'misaligned'
        p = P.argmax(1); rows.append((tag, s, 100 * (p == y).mean(), 100 * f1_score(y, p, average='macro')))
R = pd.DataFrame(rows, columns=['head', 'seed', 'val acc', 'val macro-F1']); print(R.round(2).to_string(index=False))
from scipy import stats
for tag, g in R.groupby('head'):
    f = g['val macro-F1'].values; a = g['val acc'].values
    ci = lambda v: stats.t.ppf(.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
    print(f"{tag}: val accuracy {a.mean():.2f} +/- {ci(a):.2f} | val macro-F1 {f.mean():.2f} +/- {ci(f):.2f}")